# Aeromagnetic data QC

This notebook demonstrates the use of functions to perform QC of aeromagnetic data.

Here we demonstrate their use on some aeromagnetic data from the Eastern Victoria survey.

Ensure you have run the `Prepare_AeromagData` notebook first so that the data are prepared for review.

___

First, import the required python packages, ...

In [ ]:
from pathlib import Path
import galileoQC as qc

... then set the path to the geowhizz files.

This is all very much step by step to illustrate the process, and you can certainly compress some of these steps in your own work.

In [ ]:
dx = Path(r'./MagData/FD013_Mag.xyz')
dh = dx.with_suffix(".hdf5")

In [ ]:
if not dh.exists():
    %run ./Prepare_AeromagData.ipynb

We will need channel names later so let's report them now.

In [ ]:
qc.reportChannels(dh, verbose=True)

___

A standard technical specification is the requirement that the basemag diurnal monitoring data, collected while each line is sampled, not deviate from a straight-line chord of a certain time span by more than a given number of nanoTesla.

The `checkDiurnal` function checks the data for such deviations in the given channel, deviations being less than `rangeLimit` over chords whose time span is `nSamples` long (as number of fids or samples).

In [ ]:
qc.checkDiurnal(dh, 'DIURNAL', lines=[],
                rangeLimit = 0.02, nSamples = 8, plot_flag=True)

___

The standard check for noise in aeromagnetic data is difference analysis. Very high frequency signal is assumed to be noise and this is measured by differencing the signal. Commonly the 4-th difference or 8-th difference is used.

The function `checkDiff` checks the total magnetic field n-th difference channel in a whizzFile against the specification that the peak to peak variation over a set number of samples must not exceed some peak value. If no n-th difference channel is supplied, then the raw magnetic data channel may be used, and its n-th difference will be calculated by the function.

The `GA Magnetic Deed` sets a specification:

    Magnetic noise on unfiltered magnetic data acquired during the survey must be less than 0.1 nT.
    The 8th difference will be used to assess the magnetic noise. If the signal of the 8th difference
    exceeds 0.1 nT over a distance of 1000 m the line will be re-flown at the contractor’s expense.
    
For a joint gravity-magnetic survey n-th difference noise is typically higher than for a solely magnetometer survey and technical specifications might be relaxed. 

The `plot_flag` and `verbose` Boolean parameters control whether the results are plotted, and whether a verbose report is given, as usual.

In [ ]:
qc.checkDiff(dh, diff_chan='', raw_chan='LVLDMAG',
             lines=[],
             limit=0.001, nSamples=300, num_diff=8,
             plot_flag=True, verbose=True)

In [ ]:
qc.checkDiff(dh, diff_chan='', raw_chan='LVLDMAG',
             lines=[],
             limit=0.001, maxDuration=30.0, num_diff=8,
             plot_flag=True, verbose=True)

In [ ]:
qc.checkDiff(dh, diff_chan='', raw_chan='LVLDMAG',
             lines=[],
             limit=0.001, maxDistance=1800.0, num_diff=8,
             plot_flag=True, verbose=True)

___

It is useful to check the power spectrum of the magnetic data because periodic noise sources can occur and these are often not seen in the standard 4th difference measure. This can be performed at each stage of processing, or just on the raw magnetic data and the final magnetic data. The power spectrum should not have any signs of harmonic signal.

In the example below, there are no spikes in the spectrum above $10^{-3}$ so all okay.

In [ ]:
qc.psdChannel(dh, 'COMPMAG', flightLines=[], shortestPeriod=0.0, minlinelenkm=15.0, verbose=False)